# Non-Ergodic
In non-ergodic systems, the mean growth does not equal the median growth. Peters' coin toss is the perfect example. In it, the expected equity grows over time by 5%, whilst the time-averaged equity actually declines at the same rate! A positive expectation does not necessarily mean you will make money if you do not try to maximise the time-average growth rate.

So what fraction of our wealth should we bet? The Kelly criterion maximises the growth rate, but in real life, we almost never know our payoff / probability of winning (or volatility / excess return) well enough to use it. Uncertainty in our estimates should make us shrink our risk, but by how much?

**Author:** puzzle

**Created:** 2026-03-13

**Modified:** 2026-03-13

### Sources

- [1] Peters, O. (2023) The Infamous Coin Toss. Available at: https://ergodicityeconomics.com/2023/07/28/the-infamous-coin-toss/ (Accessed 13th Mar 2026).
- [2] Peters, O. (2025) Expected-Utility Maximizers Don’t Maximize Utility. Available at: https://ergodicityeconomics.com/2025/05/28/expected-utility-maximizers-dont-maximize-utility/ (Accessed 13th Mar 2026).
- [3] Breiman, L. (1961) Optimal Gambling Systems for Favorable Games. Proceedings of the 4th Berkeley Symposium on Mathematical Statistics and Probability, 1, 65-78.
- [4] Peters, O. (2019) The Ergodicity Problem in Economics. Nature Physics, 15, 1216–1221.
- [5] Kelly, J. L. (1956) A New Interpretation of Information Rate. Bell System Technical Journal, 35, 917-926. Available at: https://www.princeton.edu/~wbialek/rome/refs/kelly_56.pdf (Accessed 13th Mar 2026).
- [6] Browne, S. & Ward, W. (1996) Portfolio Choice and the Bayesian Kelly Criterion. Advances in Applied Probability, 28(4), 1145-1176. Available at: https://www.columbia.edu/~ww2040/PortfolioChoice96.pdf (Accessed 13th Mar 2026).


In [1]:
from itertools import product

In [2]:
import numpy as np
import pandas as pd
import plotly.colors as pc
import plotly.express as px
import plotly.graph_objects as go

In [3]:
from jfmi.plot.utilities import load_plotly_templates, unparse_rgba_tuple

In [4]:
from jfmi.plot.templates import COLOURS

In [5]:
load_plotly_templates()

### Simulate Bets


In [6]:
n_simulations = 1000
n_bets = 50

b = 0.5  # 0.8  # return win
a = 0.4  # 0.5  # return loss
p = 0.5  # probability win
q = 1 - p  # probability loss

risk = 1.0  # fraction or fixed size
# risk = p / a - q / b  # optimal (Kelly) risk fraction

p * b + q * (-a)  # expected value

0.04999999999999999

In [7]:
# Start with an initial account equity of 1.
arr_paths = np.ones((n_simulations, n_bets + 1))

arr_bets = np.random.rand(n_simulations, n_bets) < p

#### Multiplicative / Geometric


In [8]:
arr_multiples = np.where(arr_bets, 1 + risk * b, 1 - risk * a)

# Simulate equity paths.
for t in range(1, n_bets + 1):
    arr_paths[:, t] = (
        arr_paths[:, t - 1] * arr_multiples[:, t - 1]
    )  # multiplicative/geometric

#### Additive / Linear

In [9]:
# # Simulate equity paths.
# for t in range(1, n_bets + 1):
#     # The bet size cannot exceed the current equity.
#     bet = np.minimum(risk, arr_paths[:, t - 1])

#     arr_paths[:, t] = arr_paths[:, t - 1] + np.where(
#         arr_bets[:, t - 1], bet * b, -bet * a
#     )  # additive/linear

#     # Stop the simulation if equity hits 0.
#     arr_paths[:, t] = np.maximum(arr_paths[:, t], 0)

### Calculate Mean and Median


In [10]:
mean_equity = arr_paths.mean(axis=0)
std_equity = arr_paths.std(axis=0)

mean_equity_standard_error = std_equity / np.sqrt(n_simulations)

In [11]:
def bootstrap_percentile_ci(data, percentile, n_boot=1000):
    """Bootstrap percentile confidence intervals.

    Randomly resample the data with replacement to compute the statistic lots of times,
    then take the 95% confidence intervals of the percentiles. This can be useful when
    distributions are non-normal and analytical solutions are difficult.
    """
    point = np.percentile(data, percentile)

    boot = []

    for _ in range(n_boot):
        sample = np.random.choice(data, size=len(data), replace=True)
        boot.append(np.percentile(sample, percentile))

    # Take the 95% confidence interval.
    ci_low, ci_high = np.percentile(boot, [2.5, 97.5])

    return point, ci_low, ci_high

In [12]:
median_equity = np.zeros(n_bets + 1)
median_equity_ci_lows = np.zeros(n_bets + 1)
median_equity_ci_highs = np.zeros(n_bets + 1)

for t in range(n_bets + 1):
    median, ci_low, ci_high = bootstrap_percentile_ci(arr_paths[:, t], 50)

    median_equity[t] = median
    median_equity_ci_lows[t] = ci_low
    median_equity_ci_highs[t] = ci_high

### Plot Mean and Median Equities


In [13]:
arr_bets = np.arange(n_bets + 1)

fig = go.Figure()

# Plot the mean equity.
fig.add_trace(
    go.Scatter(
        x=arr_bets,
        y=mean_equity,
        mode="lines",
        line=dict(color=COLOURS["blue"][5]),
        name="Mean equity",
    )
)

# Plot the mean equity error band.
fig.add_trace(
    go.Scatter(
        x=np.concatenate([arr_bets, arr_bets[::-1]]),
        y=np.concatenate(
            [
                mean_equity + mean_equity_standard_error,
                (mean_equity - mean_equity_standard_error)[::-1],
            ]
        ),
        fill="toself",
        line=dict(color="rgba(0,0,0,0)"),
        fillcolor=unparse_rgba_tuple((*pc.hex_to_rgb(COLOURS["blue"][5]), 0.2)),
        name="Mean Error",
    )
)

# Plot the theoretical median equity.
fig.add_trace(
    go.Scatter(
        x=arr_bets,
        y=np.exp(arr_bets * (p * np.log(1 + risk * b) + q * np.log(1 - risk * a))),
        mode="lines",
        line=dict(color=COLOURS["orange"][5]),
        name="Theoretical Median equity",
    )
)

# Plot the median equity.
fig.add_trace(
    go.Scatter(
        x=arr_bets,
        y=median_equity,
        mode="lines",
        line=dict(color=COLOURS["purple"][5]),
        name="Median equity",
    )
)

# Plot the median equity error band.
fig.add_trace(
    go.Scatter(
        x=np.concatenate([arr_bets, arr_bets[::-1]]),
        y=np.concatenate([median_equity_ci_highs, median_equity_ci_lows[::-1]]),
        line=dict(color="rgba(0,0,0,0)"),
        fill="toself",
        fillcolor=unparse_rgba_tuple((*pc.hex_to_rgb(COLOURS["purple"][5]), 0.2)),
        name="Median 95% CI",
    )
)

fig.update_layout(
    title="Mean vs. Median Equity",
    xaxis_title="Number of Tosses",
    yaxis_title="Equity",
    yaxis_type="log",
    margin=dict(t=80),
)

fig.show()

### Find All Possible Paths


In [14]:
n_bets = 10

arr_bets = np.arange(n_bets + 1)

#### Multiplicative / Geometric


In [15]:
paths = []

# Where 1 is a win, and 0 is a loss.
for sequence in product([1, 0], repeat=n_bets):
    # Start with an initial account equity of 1.
    path = [1]

    for outcome in sequence:
        multiple = 1 + risk * b if outcome else 1 - risk * a
        path.append(path[-1] * multiple)  # multiplicative/geometric

    paths.append(path)

paths = np.array(paths)  # shape (2**n_bets, n_bets+1)

#### Additive / Linear


In [16]:
# paths = []

# # Where 1 is a win, and 0 is a loss.
# for sequence in product([1, 0], repeat=n_bets):
#     # Start with an initial account equity of 1.
#     path = [1]

#     for outcome in sequence:
#         equity = path[-1]

#         # The bet size cannot exceed the current equity.
#         bet = min(risk, equity)
#         # bet = risk

#         # Stop the simulation if equity hits 0.
#         if equity <= 0:
#             path.append(0)

#             continue

#         addition = equity + bet*b if outcome else max(equity - bet*a, 0)
#         path.append(addition)  # additive/linear

#     paths.append(path)

# paths = np.array(paths)  # shape (2**n_bets, n_bets+1)

### Count Node Frequency


In [17]:
# Count how many times each node has been reached at each bet.
node_counts = []

for t in range(n_bets + 1):
    unique, counts = np.unique(paths[:, t], return_counts=True)
    node_counts.append(dict(zip(unique, counts, strict=True)))

In [18]:
arr_mean_equity = paths.mean(axis=0)
arr_median_equity = np.median(paths, axis=0)

### Plot All Possible Paths


In [19]:
fig = go.Figure()

for path in paths:
    hover_text = [
        f"Toss: {t}<br>"
        f"Equity: {equity:.3f}<br>"
        f"Reached: {node_counts[bet][equity]} times"
        for bet, equity in enumerate(path)
    ]

    fig.add_trace(
        go.Scatter(
            x=arr_bets,
            y=path,
            mode="lines",
            opacity=0.1,
            hoverinfo="text",
            hovertext=hover_text,
            showlegend=False,
        )
    )

# Plot the mean equity.
fig.add_trace(
    go.Scatter(
        x=arr_bets,
        y=arr_mean_equity,
        mode="lines",
        line=dict(color=COLOURS["blue"][5], width=3),
        hovertemplate=(
            "<b>Toss:</b> %{x:.2f}<br>"
            "<b>Mean Equity:</b> %{y:.2f}<br>"
            "<extra></extra>"
        ),
        name="Mean Equity",
    )
)

# Plot the median equity.
fig.add_trace(
    go.Scatter(
        x=arr_bets,
        y=arr_median_equity,
        mode="lines",
        line=dict(color=COLOURS["orange"][5], width=3),
        hovertemplate=(
            "<b>Toss:</b> %{x:.2f}<br>"
            "<b>Median Equity:</b> %{y:.2f}<br>"
            "<extra></extra>"
        ),
        name="Median Equity",
    )
)

fig.update_layout(
    title="All Possible Equity Paths",
    xaxis_title="Coin Toss",
    yaxis_title="Equity",
    yaxis_type="log",
    margin=dict(t=80),
)

fig.show()